In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/digit-recognizer/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/digit-recognizer/test.csv')

X = train.drop('label', axis=1).values / 255.0
y = to_categorical(train['label'].values)
X_test = test.values / 255.0

X = X.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

In [ ]:
model = keras.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=10, validation_data=(X_val, y_val), batch_size=64)

In [ ]:
preds = np.argmax(model.predict(X_test), axis=1)

submission = pd.DataFrame({'ImageId': range(1, len(preds)+1), 'Label': preds})
submission.to_csv('submission.csv', index=False)
submission.head()

In [ ]:
idx = 2

img = X_val[idx].reshape(28, 28)
actual = np.argmax(y_val[idx])
predicted = np.argmax(model.predict(X_val[idx:idx+1], verbose=0))
confidence = model.predict(X_val[idx:idx+1], verbose=0)[0][predicted] * 100

plt.imshow(img, cmap='gray')
plt.title(f'Position {idx} | Actual: {actual} | Predicted: {predicted} ({confidence:.1f}%)', fontsize=10)
plt.axis('off')
plt.show()

print(f'Actual digit     : {actual}')
print(f'Predicted digit  : {predicted}')
print(f'Confidence       : {confidence:.1f}%')
print(f'Correct?         : {"YES" if actual == predicted else "NO"}')